# URBAN PULSE: 02 - Data Preparation, Quality Validation & Feature Engineering
### Domain: Public-Sector Service Management
This notebook cleans raw NYC 311 service request records, evaluates data quality, performs text and temporal normalization, engineers key operational variables, screens outliers, and compiles the data dictionary.

**Workflow:**
1. Load & inspect raw dataset
2. Deduplicate primary key (`unique_key`)
3. Parse timestamps to NYC local time (`America/New_York`)
4. Normalize category text & standard borough mappings
5. Derive `resolution_hours`, `is_closed`, and temporal components
6. Screen resolution duration outliers using the IQR rule
7. Export `data/cleaned_data.csv`, `data/resolution_outliers.csv`, and Data Dictionary


In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd

sys.path.append("..")
from src.cleaning import inspect_raw_data, clean_and_transform, generate_data_dictionary


### 1. Load Raw Dataset & Inspect Initial Missingness


In [2]:
raw_path = "../data/raw_data.csv"
df_raw = pd.read_csv(raw_path, low_memory=False)
inspection = inspect_raw_data(df_raw)
print(f"Total Rows: {inspection['initial_rows']:,}")
print(f"Total Missing Cells: {inspection['total_missing_cells']:,}")
print(f"Duplicate Unique Keys: {inspection['duplicate_unique_keys']}")


[STEP 2] Raw Inspection Summary:
         - Total rows: 81,115
         - Duplicate unique keys: 0
         - Total missing values: 189,547
Total Rows: 81,115
Total Missing Cells: 189,547
Duplicate Unique Keys: 0


### 2. Execute Data Cleaning & Quality Pipeline


In [3]:
df_clean, cleaning_log, outliers_df = clean_and_transform(df_raw)
print(f"Cleaned records count: {len(df_clean):,}")
print(f"Valid closed records: {cleaning_log['closed_records_count']:,}")
print(f"Open/In-progress records: {cleaning_log['open_records_count']:,}")
print(f"IQR Outliers flagged: {len(outliers_df):,}")


[WARN] Found 73 records with negative resolution hours (closed_date < created_date).
Cleaned records count: 81,115
Valid closed records: 63,484
Open/In-progress records: 17,631
IQR Outliers flagged: 9,458


### 3. Review Engineered Features & Statistical Summary
Key derived fields:
- `resolution_hours`: Elapsed hours between `created_date` and `closed_date`
- `is_closed`: Boolean closure status
- `created_year`, `created_month`, `created_weekday`, `created_hour`
- `duration_category`: `< 24h`, `24-72h`, `> 72h`


In [4]:
print(df_clean[["created_date", "closed_date", "resolution_hours", "is_closed", "duration_category"]].head(5))
print("\nResolution Hours Descriptive Statistics:")
print(df_clean[df_clean["is_valid_resolution"]]["resolution_hours"].describe())


         created_date         closed_date  ...  is_closed duration_category
0 2026-09-11 22:07:12 2026-09-12 00:30:56  ...       True    Under 24 Hours
1 2026-09-12 10:38:03 2026-09-12 10:54:21  ...       True    Under 24 Hours
2 2026-09-12 15:08:46 2026-09-12 16:18:38  ...       True    Under 24 Hours
3 2026-09-12 09:49:22 2026-09-12 10:48:48  ...       True    Under 24 Hours
4 2026-09-12 13:55:16 2026-09-12 15:53:01  ...       True    Under 24 Hours

[5 rows x 5 columns]

Resolution Hours Descriptive Statistics:
count    63411.000000
mean        68.036649
std        854.600610
min          0.000000
25%          0.787500
50%          2.822778
75%         23.938056
max      52020.854444
Name: resolution_hours, dtype: float64


### 4. Review Statistical Outlier Screening Summary
Screening using $Q1 - 1.5 \times IQR$ and $Q3 + 1.5 \times IQR$.


In [5]:
stats = cleaning_log.get("iqr_statistics", {})
for k, v in stats.items():
    print(f"  {k}: {v}")


  valid_resolution_records: 63411
  mean_hours: 68.04
  median_hours: 2.82
  std_hours: 854.6
  skewness: 31.732
  q1_25th_percentile: 0.79
  q3_75th_percentile: 23.94
  iqr: 23.15
  lower_bound: 0.0
  upper_bound: 58.66
  outlier_count: 9458
  outlier_percentage: 14.92


### 5. Generate and Verify Field-Level Data Dictionary


In [6]:
dict_md, dict_records = generate_data_dictionary(df_clean)
dict_df = pd.DataFrame(dict_records)
print(dict_df[["field_name", "data_type", "null_count", "null_percentage"]].head(10))


                       field_name       data_type  null_count  null_percentage
0                      unique_key           int64           0             0.00
1                    created_date  datetime64[us]           0             0.00
2                     closed_date  datetime64[us]       17631            21.74
3  resolution_action_updated_date  datetime64[us]        7247             8.93
4                          agency             str           0             0.00
5                     agency_name             str           0             0.00
6                  complaint_type             str           0             0.00
7                      descriptor             str        1384             1.71
8                   location_type             str        5916             7.29
9                    incident_zip         float64         754             0.93
